In [1]:
import json

In [4]:
g = json.loads("graph-data.js")

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [6]:
import re
import json

# Replace 'your_file.js' with the actual path to your file
file_path = 'graph-data.js' 

with open(file_path, 'r', encoding='utf-8') as f:
    raw_content = f.read().strip()

# 1. Strip the 'const GRAPH_DATA = ' from the start and the trailing ';' if present
# This extracts everything between the first '{' and the last '}'
json_clean_match = re.search(r'\{.*\}', raw_content, re.DOTALL)

if json_clean_match:
    json_string = json_clean_match.group(0)
    
    # 2. Parse the pure string as a standard Python dictionary
    GRAPH_DATA = json.loads(json_string)
    print("✓ File successfully loaded and cleaned into JSON format!")
    
    # 3. Execute the key discovery logic
    enzyme_root_keys = set()
    tissue_weight_keys = set()

    for node in GRAPH_DATA.get("nodes", []):
        if node.get("type") == "Enzyme":
            enzyme_root_keys.update(node.keys())
            
            tissue_data = node.get("tissue_weights")
            if isinstance(tissue_data, dict):
                tissue_weight_keys.update(tissue_data.keys())

    print("\n--- All Keys Encountered on 'Enzyme' Nodes ---")
    print(sorted(list(enzyme_root_keys)))

    print("\n--- All Inner Keys Encountered inside 'tissue_weights' ---")
    print(sorted(list(tissue_weight_keys)))

else:
    print("❌ Error: Could not find valid JSON boundaries within the file structure.")

✓ File successfully loaded and cleaned into JSON format!

--- All Keys Encountered on 'Enzyme' Nodes ---
['activity_score', 'canonical_id', 'canonical_label', 'canonical_namespace', 'detail', 'evidence', 'group', 'id', 'label', 'match_status', 'origin', 'phase', 'provenance', 'role', 'source_db', 'tier', 'tissue', 'tissue_weights', 'type', 'variant']

--- All Inner Keys Encountered inside 'tissue_weights' ---
['Bladder', 'Breast', 'Colon', 'Esophagus', 'Kidney', 'Liver', 'Lung', 'Prostate']


In [7]:
# 1. Collect key structures across all Enzyme nodes
enzyme_nodes = [node for node in GRAPH_DATA.get("nodes", []) if node.get("type") == "Enzyme"]

if not enzyme_nodes:
    print("No Enzyme nodes found to compare.")
else:
    # 2. Extract the set of keys from the very first enzyme to initialize arrays
    all_encountered_keys = set(enzyme_nodes[0].keys())
    keys_in_every_enzyme = set(enzyme_nodes[0].keys())

    # 3. Intersect and union keys across the rest of the nodes
    for node in enzyme_nodes[1:]:
        current_keys = set(node.keys())
        all_encountered_keys.update(current_keys)          # Grows to include everything
        keys_in_every_enzyme.intersection_update(current_keys) # Shrinks to only shared keys

    # 4. Inconsistent keys are the difference between the two sets
    optional_keys = all_encountered_keys - keys_in_every_enzyme

    print(f"Total Enzyme nodes analyzed: {len(enzyme_nodes)}")
    print(f"► Core keys (In ALL enzymes): {sorted(list(keys_in_every_enzyme))}")
    print(f"► Optional keys (In SOME but not all): {sorted(list(optional_keys))}\n")

    # 5. Print a quick breakdown of where the optional keys appear
    if optional_keys:
        print("--- Optional Keys Breakdown ---")
        for key in sorted(list(optional_keys)):
            count = sum(1 for n in enzyme_nodes if key in n)
            print(f"Key '{key}' is present in {count}/{len(enzyme_nodes)} enzymes.")

Total Enzyme nodes analyzed: 60
► Core keys (In ALL enzymes): ['detail', 'id', 'label', 'match_status', 'origin', 'provenance', 'role', 'type']
► Optional keys (In SOME but not all): ['activity_score', 'canonical_id', 'canonical_label', 'canonical_namespace', 'evidence', 'group', 'phase', 'source_db', 'tier', 'tissue', 'tissue_weights', 'variant']

--- Optional Keys Breakdown ---
Key 'activity_score' is present in 27/60 enzymes.
Key 'canonical_id' is present in 37/60 enzymes.
Key 'canonical_label' is present in 37/60 enzymes.
Key 'canonical_namespace' is present in 37/60 enzymes.
Key 'evidence' is present in 37/60 enzymes.
Key 'group' is present in 8/60 enzymes.
Key 'phase' is present in 49/60 enzymes.
Key 'source_db' is present in 45/60 enzymes.
Key 'tier' is present in 45/60 enzymes.
Key 'tissue' is present in 54/60 enzymes.
Key 'tissue_weights' is present in 23/60 enzymes.
Key 'variant' is present in 13/60 enzymes.


In [8]:
from collections import defaultdict

# 1. Group raw nodes by their "type" field
nodes_by_type = defaultdict(list)
for node in GRAPH_DATA.get("nodes", []):
    node_type = node.get("type", "Unknown")
    nodes_by_type[node_type].append(node)

# 2. Track key sets for each node type
all_keys_by_type = {}
common_keys_by_type = {}

for node_type, nodes in nodes_by_type.items():
    # Initialize with the keys of the first node in this type group
    type_all_keys = set(nodes[0].keys())
    type_common_keys = set(nodes[0].keys())
    
    # Intersect and union across all remaining nodes of this specific type
    for node in nodes[1:]:
        current_keys = set(node.keys())
        type_all_keys.update(current_keys)
        type_common_keys.intersection_update(current_keys)
        
    all_keys_by_type[node_type] = type_all_keys
    common_keys_by_type[node_type] = type_common_keys

# 3. Calculate Global Intersection (Keys common to ALL nodes regardless of type)
global_common_keys = None
for nodes in nodes_by_type.values():
    for node in nodes:
        if global_common_keys is None:
            global_common_keys = set(node.keys())
        else:
            global_common_keys.intersection_update(set(node.keys()))
global_common_keys = global_common_keys or set()

# --- PRINT ANALYSIS RESULTS ---

print("==================================================")
print(" GLOBAL ANALYSIS (ALL NODE TYPES)")
print("==================================================")
print(f"▶ Keys common to ALL nodes across the entire dataset:\n  {sorted(list(global_common_keys))}\n")

print("==================================================")
print(" TYPE-SPECIFIC ANALYSIS")
print("==================================================")

for node_type in sorted(nodes_by_type.keys()):
    nodes = nodes_by_type[node_type]
    type_all = all_keys_by_type[node_type]
    type_common = common_keys_by_type[node_type]
    
    # Optional keys within this type: fields present in some, but not all nodes of this type
    optional_keys = type_all - type_common
    
    # Unique keys: fields that exist ONLY in this node type and nowhere else in the dataset
    other_types_keys = set()
    for other_type, keys in all_keys_by_type.items():
        if other_type != node_type:
            other_types_keys.update(keys)
    unique_keys = type_all - other_types_keys
    
    print(f"\n🔹 NODE TYPE: '{node_type}' ({len(nodes)} records found)")
    print(f"  ┌─ Core keys (In ALL '{node_type}' nodes):")
    print(f"  │  {sorted(list(type_common))}")
    print(f"  ├─ Optional keys (In SOME but not all '{node_type}' nodes):")
    print(f"  │  {sorted(list(optional_keys)) if optional_keys else 'None'}")
    print(f"  └─ Exclusive keys (ONLY found in '{node_type}' nodes):")
    print(f"     {sorted(list(unique_keys)) if unique_keys else 'None'}")


 GLOBAL ANALYSIS (ALL NODE TYPES)
▶ Keys common to ALL nodes across the entire dataset:
  ['detail', 'id', 'label', 'match_status', 'origin', 'provenance', 'type']

 TYPE-SPECIFIC ANALYSIS

🔹 NODE TYPE: 'Carcinogen' (56 records found)
  ┌─ Core keys (In ALL 'Carcinogen' nodes):
  │  ['detail', 'group', 'id', 'label', 'match_status', 'origin', 'provenance', 'type']
  ├─ Optional keys (In SOME but not all 'Carcinogen' nodes):
  │  ['canonical_id', 'canonical_label', 'canonical_namespace', 'evidence', 'exposure', 'iarc', 'source_db']
  └─ Exclusive keys (ONLY found in 'Carcinogen' nodes):
     ['exposure', 'iarc']

🔹 NODE TYPE: 'DNA_Adduct' (27 records found)
  ┌─ Core keys (In ALL 'DNA_Adduct' nodes):
  │  ['detail', 'id', 'label', 'match_status', 'origin', 'provenance', 'type']
  ├─ Optional keys (In SOME but not all 'DNA_Adduct' nodes):
  │  ['canonical_id', 'canonical_label', 'canonical_namespace', 'evidence', 'reactivity', 'source_db']
  └─ Exclusive keys (ONLY found in 'DNA_Adduct' 

In [9]:
from collections import defaultdict

# 1. Group raw nodes by their "type" field
nodes_by_type = defaultdict(list)
for node in GRAPH_DATA.get("edges", []):
    node_type = node.get("type", "Unknown")
    nodes_by_type[node_type].append(node)

# 2. Track key sets for each node type
all_keys_by_type = {}
common_keys_by_type = {}

for node_type, nodes in nodes_by_type.items():
    # Initialize with the keys of the first node in this type group
    type_all_keys = set(nodes[0].keys())
    type_common_keys = set(nodes[0].keys())
    
    # Intersect and union across all remaining nodes of this specific type
    for node in nodes[1:]:
        current_keys = set(node.keys())
        type_all_keys.update(current_keys)
        type_common_keys.intersection_update(current_keys)
        
    all_keys_by_type[node_type] = type_all_keys
    common_keys_by_type[node_type] = type_common_keys

# 3. Calculate Global Intersection (Keys common to ALL nodes regardless of type)
global_common_keys = None
for nodes in nodes_by_type.values():
    for node in nodes:
        if global_common_keys is None:
            global_common_keys = set(node.keys())
        else:
            global_common_keys.intersection_update(set(node.keys()))
global_common_keys = global_common_keys or set()

# --- PRINT ANALYSIS RESULTS ---

print("==================================================")
print(" GLOBAL ANALYSIS (ALL NODE TYPES)")
print("==================================================")
print(f"▶ Keys common to ALL nodes across the entire dataset:\n  {sorted(list(global_common_keys))}\n")

print("==================================================")
print(" TYPE-SPECIFIC ANALYSIS")
print("==================================================")

for node_type in sorted(nodes_by_type.keys()):
    nodes = nodes_by_type[node_type]
    type_all = all_keys_by_type[node_type]
    type_common = common_keys_by_type[node_type]
    
    # Optional keys within this type: fields present in some, but not all nodes of this type
    optional_keys = type_all - type_common
    
    # Unique keys: fields that exist ONLY in this node type and nowhere else in the dataset
    other_types_keys = set()
    for other_type, keys in all_keys_by_type.items():
        if other_type != node_type:
            other_types_keys.update(keys)
    unique_keys = type_all - other_types_keys
    
    print(f"\n🔹 NODE TYPE: '{node_type}' ({len(nodes)} records found)")
    print(f"  ┌─ Core keys (In ALL '{node_type}' nodes):")
    print(f"  │  {sorted(list(type_common))}")
    print(f"  ├─ Optional keys (In SOME but not all '{node_type}' nodes):")
    print(f"  │  {sorted(list(optional_keys)) if optional_keys else 'None'}")
    print(f"  └─ Exclusive keys (ONLY found in '{node_type}' nodes):")
    print(f"     {sorted(list(unique_keys)) if unique_keys else 'None'}")


 GLOBAL ANALYSIS (ALL NODE TYPES)
▶ Keys common to ALL nodes across the entire dataset:
  ['match_status', 'origin', 'provenance', 'source', 'target', 'type']

 TYPE-SPECIFIC ANALYSIS

🔹 NODE TYPE: 'ACTIVATES' (104 records found)
  ┌─ Core keys (In ALL 'ACTIVATES' nodes):
  │  ['match_status', 'origin', 'provenance', 'source', 'target', 'type']
  ├─ Optional keys (In SOME but not all 'ACTIVATES' nodes):
  │  ['canonical_namespace', 'canonical_predicate', 'carcinogen', 'evidence', 'label', 'source_db']
  └─ Exclusive keys (ONLY found in 'ACTIVATES' nodes):
     None

🔹 NODE TYPE: 'DETOXIFIES' (47 records found)
  ┌─ Core keys (In ALL 'DETOXIFIES' nodes):
  │  ['carcinogen', 'match_status', 'origin', 'provenance', 'source', 'target', 'type']
  ├─ Optional keys (In SOME but not all 'DETOXIFIES' nodes):
  │  ['canonical_namespace', 'canonical_predicate', 'evidence', 'label', 'source_db']
  └─ Exclusive keys (ONLY found in 'DETOXIFIES' nodes):
     None

🔹 NODE TYPE: 'FORMS_ADDUCT' (37 reco